# PRÉ-PROJETO DE PESQUISA OPERACIONAL — Pyomo com HiGHS, CBC ou Gurobi

## Otimização Operacional do Agendamento de Bombas e Distribuição de Água Tratada via Programação Linear Inteira Mista (MILP)

- Escolha o solver alterando apenas a variável `SOLVER` na célula de configuração.
- A célula inicial instala Pyomo, HiGHS, CBC e a interface Python do Gurobi.

In [ ]:
%pip install --upgrade "pyomo>=6.8" "highspy>=1.8" cbcbox gurobipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyomo.environ import (
    ConcreteModel, Set, Var, Param, Objective, Constraint, Binary,
    NonNegativeReals, RangeSet, minimize, value, SolverFactory, TerminationCondition
)
import pyomo.environ as pyo

# 1. Conjuntos de Índices

T = 24
horas = range(T)
bombas = range(1, 7)
reservatorios = range(1, 6)
setores = range(1, 13)

# 2. Topologia do Sistema (Arcos permitidos conforme o diagrama)
# Arcos (Bomba -> Reservatório)
arcos_PR = [
    (1,1), 
    (2,1), (2,2), (2,3), 
    (3,2), 
    (4,3), (4,4), 
    (5,5), 
    (6,4), (6,5)
]

# Arcos (Reservatório -> Setor de Consumo)
arcos_RD = [
    (1,1), (1,2), (1,3),
    (2,2), (2,4), (2,5),
    (3,4), (3,6), (3,7),
    (4,4), (4,8), (4,9), (4,10), (4,12),
    (5,7), (5,10), (5,11), (5,12)
]

# 3. Parâmetros Econômicos (Tarifa Horossazonal - R$/kWh)
# Tarifa fora de ponta: R$ 0.4525 | Tarifa de ponta (18h00 às 20h59 ): R$ 1.4048
C = {t: 1.4048 if 18 <= t < 21 else 0.4525 for t in horas}

# 4. Parâmetros das bombas: (potência de referência em kW, vazão mínima e máxima em m³/h)
# Potências estimadas no ponto de vazão máxima, com 30 m de altura manométrica e 80% de eficiência global.
# O objetivo considera essa potência constante durante cada período ligado.
param_bombas = {
    1: (240, 1170, 2340),
    2: (450, 2180, 4370),
    3: (240, 1170, 2340),
    4: (290, 1400, 2810),
    5: (190,  940, 1870),
    6: (240, 1170, 2340)
}
Q_ETA_max = 9400  # m³/h; limite de vazão agregada da ETA

# 5. Parâmetros dos Reservatórios: (volume mínimo, máximo e inicial), em m³
# O volume mínimo é a reserva de segurança; o volume inicial define o estoque no início do horizonte.
param_res = {
    1: (2100, 14000, 4200),
    2: (1400,  9400, 2800),
    3: (1400,  9400, 2800),
    4: (1400,  9400, 2800),
    5: ( 700,  4700, 1400)
}

# 6. Demandas-base dos setores (m³/h); total: 5195 m³/h
base_demanda = {
    1: 312,  2: 468,  3: 390,  4: 624,
    5: 390,  6: 312,  7: 546,  8: 468,
    9: 312, 10: 702, 11: 390, 12: 281
}

# Curva horária de demanda de verão; cada fator multiplica a demanda-base dos setores.
# Demanda média diária: 8312 m³/h; pico: 17663 m³/h às 18h.
curva_verao = [
    0.8, 0.5, 0.3, 0.4, 0.6, 1.0,   # 00h às 05h: Queda brusca do consumo na madrugada
    1.8, 2.0, 1.9, 1.7, 1.5, 1.5,   # 06h às 11h: Pico pesado da manhã
    1.4, 1.5, 1.6, 1.7, 1.8, 2.9,   # 12h às 17h: Consumo alto e contínuo
    3.4, 3.0, 2.6, 2.0, 1.5, 1.0    # 18h às 23h: Pico máximo (chegada da praia + banhos)
]

D = {}
for d in setores:
    for t in horas:
        D[d, t] = round(base_demanda[d] * curva_verao[t], 2)



In [ ]:
# Altere somente esta variável: "highs", "cbc" ou "gurobi".
SOLVER = "gurobi"

SOLVERS = {
    "highs": {
        "factory": "appsi_highs",
        "options": {
            "time_limit": 300,
            "mip_rel_gap": 0.003,
            "threads": 0,
            "random_seed": 0,
            "mip_detect_symmetry": True,
            "parallel": "off",
        },
    },
    "highs_base": {
            "factory": "appsi_highs",
            "options": {
                "time_limit": 300,
                "mip_rel_gap": 0.003,
                "threads": 0,                
            },
        },
    "cbc": {
        "factory": "cbc",
        "options": {
            "seconds": 300,
            "ratio": 0.003,
            "threads": 0,
            "randomSeed": 0,
        },
    },
    "gurobi": {
        "factory": "gurobi_direct",
        "options": {
            "TimeLimit": 300,
            "MIPGap": 0.003,
            "Threads": 0,
            "Seed": 0,
        },
    },
}

if SOLVER not in SOLVERS:
    raise ValueError(
        f"SOLVER inválido: {SOLVER!r}. Escolha entre {', '.join(SOLVERS)}."
    )

def criar_solver(verbose=False):
    config = SOLVERS[SOLVER]
    if SOLVER == "cbc":
        import cbcbox
        solver = SolverFactory(
            config["factory"], executable=cbcbox.cbc_bin_path()
        )
    else:
        solver = SolverFactory(config["factory"])
    if not solver.available(exception_flag=False):
        raise RuntimeError(
            f"O solver {SOLVER!r} não está disponível. Instale/configure o solver "
            "selecionado e, para Gurobi, verifique também a licença."
        )

    for option, setting in config["options"].items():
        solver.options[option] = setting

    if SOLVER == "highs":
        solver.options["output_flag"] = verbose
    elif SOLVER == "cbc":
        solver.options["logLevel"] = 1 if verbose else 0
    else:
        solver.options["OutputFlag"] = int(verbose)
    return solver


In [ ]:
# Criação do modelo em Pyomo
model = ConcreteModel(name="Agendamento_Bombas_MILP")

# Conjuntos
model.P = RangeSet(1, 6)
model.R = RangeSet(1, 5)
model.D = RangeSet(1, 12)
model.T = RangeSet(0, 23)
model.ARCO_PR = Set(dimen=2, initialize=arcos_PR)
model.ARCO_RD = Set(dimen=2, initialize=arcos_RD)

# Variáveis
model.x = Var(model.P, model.T, domain=Binary)
model.q = Var(model.P, model.T, domain=NonNegativeReals)
model.f_pr = Var(model.ARCO_PR, model.T, domain=NonNegativeReals)
model.f_rd = Var(model.ARCO_RD, model.T, domain=NonNegativeReals)
model.V = Var(model.R, model.T, domain=NonNegativeReals)

# Objetivo: minimizar o custo diário de energia
model.obj = Objective(
    expr=sum(C[t] * param_bombas[p][0] * model.x[p, t]
             for p in model.P for t in model.T),
    sense=minimize
)

# Limites operacionais das bombas
model.limite_min_bomba = Constraint(
    model.P, model.T,
    rule=lambda m, p, t: m.q[p, t] >= param_bombas[p][1] * m.x[p, t]
)
model.limite_max_bomba = Constraint(
    model.P, model.T,
    rule=lambda m, p, t: m.q[p, t] <= param_bombas[p][2] * m.x[p, t]
)

def regra_fluxo_bomba(m, p, t):
    arcos_saida = [arco for arco in arcos_PR if arco[0] == p]
    return m.q[p, t] == sum(m.f_pr[p, r, t] for p, r in arcos_saida)
model.fluxo_bomba = Constraint(model.P, model.T, rule=regra_fluxo_bomba)

# Capacidade máxima da ETA
model.capacidade_ETA = Constraint(
    model.T,
    rule=lambda m, t: sum(m.q[p, t] for p in model.P) <= Q_ETA_max
)

# Atendimento da demanda
model.demanda = Constraint(
    model.D, model.T,
    rule=lambda m, d, t: sum(m.f_rd[r, d, t]
                              for r, setor in arcos_RD if setor == d) == D[d, t]
)

# Balanço hídrico dos reservatórios
model.balanco = Constraint(
    model.R, model.T,
    rule=lambda m, r, t: m.V[r, t] ==
        (param_res[r][2] if t == 0 else m.V[r, t - 1])
        + sum(m.f_pr[p, r, t] for p, rr in arcos_PR if rr == r)
        - sum(m.f_rd[r, d, t] for rr, d in arcos_RD if rr == r)
)

model.V_min = Constraint(
    model.R, model.T,
    rule=lambda m, r, t: m.V[r, t] >= param_res[r][0]
)
model.V_max = Constraint(
    model.R, model.T,
    rule=lambda m, r, t: m.V[r, t] <= param_res[r][1]
)
model.recuperacao = Constraint(
    model.R,
    rule=lambda m, r: m.V[r, 23] >= param_res[r][2]
)


## Cenário-base com tarifa fixa

Este cenário mantém as mesmas demandas, capacidades e balanços dos reservatórios, mas usa uma tarifa uniforme na função objetivo. Nas horas de ponta, fixa o bombeamento total na demanda agregada da hora, limitado à capacidade máxima da ETA. Depois, o custo da programação-base é recalculado com as tarifas horárias reais. A otimização tarifária original é executada em seguida, sem alteração.

In [ ]:
# Clona o modelo para calcular uma programação sem sinal de preço horário.
# Com intervalos de 1 hora, minimizar kW * horas equivale a minimizar kWh.
modelo_base = model.clone()
modelo_base.obj.set_value(
    sum(param_bombas[p][0] * modelo_base.x[p, t]
        for p in modelo_base.P for t in modelo_base.T)
)

# Na ponta, bombeia a demanda agregada; se ela exceder a capacidade da ETA,
# usa a capacidade máxima disponível naquele horário.
horas_ponta = [t for t in horas if C[t] == max(C.values())]
vazao_base_ponta = {
    t: min(sum(D[d, t] for d in setores), Q_ETA_max)
    for t in horas_ponta
}
modelo_base.vazao_ponta = Constraint(
    horas_ponta,
    rule=lambda m, t: sum(m.q[p, t] for p in m.P) == vazao_base_ponta[t],
)

solver_base = criar_solver(verbose=False)

resultado_base = solver_base.solve(modelo_base)
status_base = resultado_base.solver.termination_condition
consumo_base_kwh = value(modelo_base.obj, exception=False)
if consumo_base_kwh is None:
    raise RuntimeError(
        f"O cenário-base não encontrou solução viável. Status: {status_base}"
    )

# Avalia a programação-base com a tarifa horária real, sem reotimizá-la.
custo_base_real = sum(
    C[t] * param_bombas[p][0] * value(modelo_base.x[p, t])
    for p in modelo_base.P for t in modelo_base.T
)
print("[CENÁRIO-BASE: TARIFA FIXA]")
print("Status:", status_base)
print("Vazão-alvo na ponta (m³/h):", vazao_base_ponta)
print(f"Consumo de energia: {consumo_base_kwh:.2f} kWh")
print(f"Custo dessa programação com as tarifas reais: R$ {custo_base_real:.2f}")

In [ ]:
# Otimização com o solver selecionado na variável SOLVER
solver = criar_solver(verbose=True)

# tee=True mostra o log do solver selecionado
import time
tempo_inicio = time.perf_counter()
resultado = solver.solve(model, tee=True)
tempo_execucao = time.perf_counter() - tempo_inicio
status_terminal = resultado.solver.termination_condition
solucao_disponivel = status_terminal in (
    TerminationCondition.optimal,
    TerminationCondition.feasible,
    TerminationCondition.maxTimeLimit,
)

limite_inferior = resultado.problem.lower_bound
incumbente = resultado.problem.upper_bound
if incumbente is None and solucao_disponivel:
    incumbente = value(model.obj)
if limite_inferior is not None and incumbente is not None and abs(incumbente) > 1e-12:
    gap_final = abs(incumbente - limite_inferior) / abs(incumbente)
else:
    gap_final = None

nos = None
try:
    tempo = float(resultado.solver.wallclock_time)
except Exception:
    tempo = tempo_execucao
print(f"\n[RESULTADOS DO {SOLVER.upper()} VIA PYOMO]")
print("Status:", status_terminal)
print(f"Custo/solução incumbente: R$ {incumbente:.2f}" if incumbente is not None else "Incumbente: não disponível")
print(f"Melhor limite inferior: R$ {limite_inferior:.2f}" if limite_inferior is not None else "Limite inferior: não disponível")
print(f"Gap relativo final: {gap_final:.2%}" if gap_final is not None else "Gap relativo final: não disponível")
print(f"Tempo: {tempo:.2f} s")
try:
    nos_relatorio = resultado.solver.statistics.branch_and_bound.number_of_created_subproblems
    if nos is None and nos_relatorio not in (None, float("inf")):
        nos = int(nos_relatorio)
except Exception:
    pass
print(f"Nós explorados: {nos}" if nos is not None else "Nós explorados: não disponível")

if solucao_disponivel:
    print(f"\n[SUCESSO] Solução encontrada pelo {SOLVER.upper()} via Pyomo!")
else:
    print(f"\nModelo inviável ou sem solução viável encontrada pelo {SOLVER.upper()}.")


In [ ]:
# Compara as duas programações usando o mesmo preço horário real.
if solucao_disponivel:
    consumo_otimizado_kwh = sum(
        param_bombas[p][0] * value(model.x[p, t])
        for p in model.P for t in model.T
    )
    custo_otimizado_real = sum(
        C[t] * param_bombas[p][0] * value(model.x[p, t])
        for p in model.P for t in model.T
    )
    comparacao_custos = pd.DataFrame([
        {
            "Cenário": "Base (tarifa fixa)",
            "Energia (kWh)": consumo_base_kwh,
            "Custo com tarifa real (R$)": custo_base_real,
        },
        {
            "Cenário": "Otimização (tarifa horária)",
            "Energia (kWh)": consumo_otimizado_kwh,
            "Custo com tarifa real (R$)": custo_otimizado_real,
        },
    ])
    display(comparacao_custos)
    economia = custo_base_real - custo_otimizado_real
    percentual_economia = 100 * economia / custo_base_real if custo_base_real else 0
    print(f"Diferença de custo: R$ {economia:.2f} ({percentual_economia:.2f}%)")
else:
    print("A comparação tarifária depende de uma solução viável da otimização.")

# GRÁFICOS E RESULTADOS

In [ ]:
# Célula 3: Extração de Resultados e Visualização (Matplotlib) - Eixo 0 a 24
if solucao_disponivel:
    import matplotlib.pyplot as plt

    # 1. Preparação dos dados
    # Eixo X da linha do tempo contínua com 25 marcos (0 a 24)
    horas_plot = list(range(25))
    
    # Vazão total bombeada (24 blocos de 1 hora)
    vazao_total = [sum(value(model.q[p, t]) for p in bombas) for t in horas]
    
    # Preço da energia (Adicionamos o último valor repetido apenas para o gráfico desenhar o degrau até o final da hora 24)
    preco_energia = [C[t] for t in horas]
    preco_energia.append(preco_energia[-1]) 

    # 2. Configuração da Figura
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    # --- GRÁFICO 1: Operação das Bombas vs Tarifa ---
    ax1.bar(horas, vazao_total, width=1, align='edge', color='#2196F3', alpha=0.8, edgecolor='white', label='Volume Bombeado (m³/h)')
    ax1.set_ylabel('Vazão Total (m³/h)', color='#0b5394', fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#0b5394')
    ax1.set_title('Perfil de Acionamento das Bombas e Custo da Energia', fontsize=14)
    ax1_twin = ax1.twinx()
    ax1_twin.step(horas_plot, preco_energia, where='post', color='#f44336', linewidth=2, label='Tarifa de Energia (R$/kWh)')
    ax1_twin.set_ylabel('Tarifa (R$/kWh)', color='#990000', fontweight='bold')
    ax1_twin.tick_params(axis='y', labelcolor='#990000')
    ax1.axvspan(17, 20, color='red', alpha=0.1) 
    
    # --- GRÁFICO 2: Comportamento dos Reservatórios (m³ Absoluto) ---
    cores_res = ['#4CAF50', '#8BC34A', '#CDDC39', '#FFEB3B', '#FFC107']
    for r in reservatorios:
        V_min, V_max, V_inicial = param_res[r]
      
        vol_absoluto = [V_inicial] + [value(model.V[r, t]) for t in horas]
        
        ax2.plot(horas_plot, vol_absoluto, marker='o', markersize=4, linewidth=2, color=cores_res[r-1], label=f'Reservatório {r}')
    
    # Sombreamento do horário de ponta
    ax2.axvspan(17, 20, color='red', alpha=0.1)
    
    ax2.set_xlabel('Hora do Dia (0 = Início do Dia, 24 = Final do Dia)', fontweight='bold')
    ax2.set_ylabel('Volume Armazenado (m³)', fontweight='bold')
    ax2.set_title('Dinâmica de Carga e Descarga das "Baterias Hídricas" (Em Valores Absolutos)', fontsize=14)
    ax2.set_xticks(range(0, 25, 2))
    ax2.set_xlim(0, 24)
    
    ax2.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
    ax1_twin.legend(loc='upper right', bbox_to_anchor=(1.25, 1.15))
    ax2.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# 3. Gráfico do Nível Geral de cada Reservatório (Em Porcentagem - 0h a 24h)
   
    horas_plot = list(range(25)) 
    
    plt.figure(figsize=(12, 6))
    cores = ['#4CAF50', '#8BC34A', '#CDDC39', '#FFEB3B', '#FFC107']
    
    for r in reservatorios:
        V_min, V_max, V_inicial = param_res[r]
        vol_absoluto = [V_inicial] + [value(model.V[r, t]) for t in horas]
        nivel_percent = [(v / V_max) * 100 for v in vol_absoluto]
        plt.plot(horas_plot, nivel_percent, marker='o', linewidth=2, color=cores[r-1], label=f'Reservatório {r}')

    # Destaque do horário de ponta: Começa na hora 17 (17:00) e termina na hora 20 (20:00)
    plt.axvspan(17, 20, color='red', alpha=0.1, label='Horário de Ponta (17h-20h)')
    
    # Linha pontilhada de referência no limite de 100% da capacidade
    plt.axhline(100, color='black', linestyle=':', alpha=0.5)
    
    plt.title('Evolução do Nível de Armazenamento por Reservatório (%)', fontsize=14)
    plt.xlabel('Hora do Dia (0 = Início do Dia, 24 = Final do Dia)', fontweight='bold')
    plt.ylabel('Volume Armazenado (%)', fontweight='bold')
    plt.xticks(range(0, 25, 2)) 
    plt.xlim(0, 24)
    plt.ylim(0, 105) 
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

else:
    print("Execute a otimização com sucesso antes de gerar os gráficos.")
